# 02 — Joinen over bronze → silver → gold

De drie zone-catalogs delen dezelfde Hive Metastore, dus Trino kan over
de zone-grens heen joinen. Dit is dezelfde patroon als de dbt-modellen in
`dbt/models/uc01..uc10/` — alleen interactief vanuit een notebook.

Wat OPA toestaat hangt af van je rol. Als de query 'denied' geeft, run
dan `SELECT current_user` (notebook 01) om te zien onder welke identiteit
je werkt.

In [ ]:
from uwv_lab import trino

# Hoeveel aanvragen per maand in silver, vs. wat we in gold hebben gerapporteerd?
trino.query('''
WITH silver_counts AS (
  SELECT date_trunc('month', aanvraag_datum) AS mnd, count(*) AS aanvragen
  FROM silver.wia.aanvraag
  GROUP BY 1
),
gold_counts AS (
  SELECT mnd, totaal_aanvragen
  FROM gold.uc01_wia_funnel
)
SELECT s.mnd, s.aanvragen AS silver_count, g.totaal_aanvragen AS gold_count
FROM silver_counts s
LEFT JOIN gold_counts g ON g.mnd = s.mnd
ORDER BY s.mnd
LIMIT 24
''')

In [ ]:
# Resultaat doorgeven aan DuckDB voor in-process aggregatie / pivot.
import duckdb
from uwv_lab import trino

df = trino.query('SELECT * FROM gold.uc01_wia_funnel LIMIT 10000')
duckdb.query("SELECT count(*), avg(totaal_aanvragen) FROM df").df()

In [ ]:
# Bronze rechtstreeks bevragen — alleen toegestaan met JIT of als platform-admin.
try:
    print(trino.query('SELECT count(*) FROM bronze.uwv.persona_created').iloc[0, 0])
except Exception as exc:
    print('Bronze geblokkeerd door OPA — verwacht voor de meeste rollen:')
    print(repr(exc))